# B2-023-generative-models-diffusion — Practice p05 — Solution

**Type:** mc · **Difficulty:** core · **Concepts:** stable-diffusion

*20 minutes.*  
**Set:** A  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Solution

**Answer: B.** The text encoder turns the prompt into context token vectors that supply the keys and values of cross-attention; the U-Net denoiser predicts $\varepsilon$ from $(z_t,t,\text{context})$ and is the only network trained during diffusion training; the VAE decoder maps the final latent, after division by the latent scaling factor, to an image; the scheduler holds the $\beta/\bar\alpha$ tables, adds noise during training (`q_sample`), and performs each reverse update during sampling (`p_sample`).

**First wrong assignment in each rejected option.**

* **A:** "the text encoder maps images to latents" — that is the VAE encoder's job; the text encoder only sees the prompt.
* **C:** "the text encoder supplies the queries of cross-attention" — ownership is reversed: the latent being denoised owns the queries, the text context owns the keys and values.
* **D:** "the text encoder, the U-Net, and the VAE are trained jointly on the diffusion loss" — the text encoder and the VAE are frozen; only the denoiser is trained.
* **E:** "the VAE decoder denoises pixels at every sampling step" — the denoiser (U-Net) works in latent space at every step; the VAE decoder runs once, at the end.

**Frozen during diffusion training:** the text encoder and the VAE (encoder and decoder). The scheduler has no trainable parameters at all (it is fixed tables and formulas). Only the U-Net denoiser, including its cross-attention blocks, is trained.

**Denoiser evaluations per guided sampling step:** **two** — one with the prompt's context ($\varepsilon_{\text{cond}}$) and one with the null/empty context ($\varepsilon_{\text{uncond}}$), combined by $\varepsilon_{\text{uncond}}+w(\varepsilon_{\text{cond}}-\varepsilon_{\text{uncond}})$. Over $T$ steps that is $2T$ evaluations.

In [ ]:
roles = {
    "text_encoder": {"produces": "context tokens", "cross_attention": "keys/values", "trained": False},
    "unet": {"produces": "eps prediction", "cross_attention": "queries", "trained": True},
    "vae": {"produces": "image from latent / latent from image", "runs_per_sample": 1, "trained": False},
    "scheduler": {"produces": "tables, q_sample, reverse update", "trained": False, "has_parameters": False},
}
frozen = sorted(name for name, info in roles.items() if not info["trained"] and name != "scheduler")
evals_per_guided_step = 2
T = 50
ANSWER = "B"
print("frozen:", frozen, " evaluations per guided step:", evals_per_guided_step, " per sample:", evals_per_guided_step * T)

### Answer check

In [ ]:
assert frozen == ["text_encoder", "vae"]
assert [name for name, info in roles.items() if info["trained"]] == ["unet"]
assert roles["text_encoder"]["cross_attention"] == "keys/values" and roles["unet"]["cross_attention"] == "queries"
assert evals_per_guided_step == 2 and evals_per_guided_step * T == 100
assert ANSWER == "B"